## Why Services exist

You finished notebook 03 with a Deployment running three nginx pods, each with its own IP. Pods come and go — every restart, rollout, and scale event mints new IPs.

Now you want a *web* pod to call your *api*. The web pod **can't**:

- **Hardcode pod IPs** — they change every rollout.
- **List all pods and filter by label** — that means watching the API server and retrying on every change: a tiny custom controller inside every client.
- **DNS-resolve `api-7c9d-x2k5p`** — there's no per-pod DNS by default, and the name carries a random suffix anyway.

What every client wants is a **stable name** and **stable address** that always point at "whichever api pods are alive right now." That's the **Service** — a tiny piece of glue between consumers and producers:

```
 web pod  --GET http://api:80-->  Service        3× api pods (IPs change)
                                     │  DNS: api → 10.96.0.42 (stable VIP)
                                     └─ kube-proxy forwards to a healthy pod IP
```

The web pod calls `http://api:80`. DNS resolves `api` to a stable virtual IP. `kube-proxy` rewrites the packet to one of the pod IPs. The web pod has no idea — and no need to know — which pod served the request, or that the api Deployment just finished a rollout.

That indirection is the entire point: **the Service decouples the client from the ever-changing set of pods behind it.** On our map, a Service is the first chip in the **Networking** box, and its job is the arrow down onto the **Pods** it fronts.